# 46. 频数图（countplot）

<!-- module-learning-arc:start -->
> **Seaborn 模块主线｜第 3 / 20 步：比较类别频数、水平与组内分布**
>
> **持续应用背景：** 开展客群消费行为差异研究：先固定样本和统计语义，再比较分布、关系和分面结果，判断差异是否稳定。
>
> **承接上一阶段：** 数据结构与主题  →  **本章任务：** 频数图（countplot）  →  **下一步：** 统计柱状图（barplot）
>
> **大作业连接：** 本章练习将成为《客群消费行为差异研究》的一部分，最终需要从样本口径和分布比较走到关系验证、分面研究与因果边界说明。
<!-- module-learning-arc:end -->


## 本章场景

分类数据到处都是——想知道哪种类别最多、占比多大，最直接的办法就是数一数每一类有多少条记录。



## 本章目标

学完本章，你将能够：

- **理解**：理解「频数图（countplot）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「频数图（countplot）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「频数图（countplot）」并读出其中的结论。


## 46.1 适用场景

**背景引入**：分类数据到处都是——想知道哪种类别最多、占比多大，最直接的办法就是数一数每一类有多少条记录。频数图（countplot）正是为这个朴素问题准备的：它不关心金额或均值，只统计每一类有多少行，一眼就能对比各类的分布，是探索分类变量时的首选。

打个比方：countplot 就像在班里数人头——它只关心'每个品类各有多少人'，至于每个人挣多少、消费多少，它一概不问。所以柱子的高低是'数量'不是'金额'，千万别把柱高读成销售额。

回答每个类别有多少条观察。


## 46.2 数据结构

一列分类变量；hue可增加第二个分类维度。


## 46.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 stat="percent" 改为 stat="count"，对比百分比与计数的纵轴含义
2. 移除 order 参数，观察无序与按频数排序的可读性差异
3. 修改 y="category" 为 x="category"，将水平柱状图改为垂直布局


## 46.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `plt.subplots()`、`sns.countplot()`、`ax.set()`、`fig.tight_layout()` | 回答每个类别有多少条观察。 | 把countplot误认为数值聚合 |
| 进阶变体 | `plt.subplots()`、`sns.countplot()`、`ax.set()`、`ax.legend()` | 在基础图表上增加分组、注释、布局或交互 | 类别顺序随数据变化 |
| 关键参数 | `order` | 类别顺序 | 把countplot误认为数值聚合 |
| 关键参数 | `hue` | 组内分类 | 类别顺序随数据变化 |
| 关键参数 | `stat` | 计数或比例 | 类别太多导致标签拥挤 |
| 关键参数 | `palette` | 颜色 | 把countplot误认为数值聚合 |


## 46.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，即使 seaborn 的 sns.set_theme
#      会重置字体，运行时也会在 set_theme 之后自动恢复。因此这里无需手动
#      import 或 addfont，直接使用即可。

# 1️⃣ 主题与数据导入：统一画风，读取三个公开数据集
sns.set_theme(style="whitegrid", context="notebook")

diamonds = pd.read_csv("/datasets/diamonds.csv")
taxis = pd.read_csv("/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
flights = pd.read_csv("/datasets/flights.csv")
print(f"Diamonds {len(diamonds):,} | Taxis {len(taxis):,} | Flights {len(flights):,} 行")


In [ ]:
# 2️⃣ 特征工程：把原始字段映射成图表统一使用的列名与派生指标
orders_full = diamonds.assign(
    category=diamonds["cut"],
    channel=diamonds["color"],
    region=diamonds["clarity"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    satisfied=np.where(
        diamonds["price"] >= diamonds["price"].median(), "高于中位价", "不高于中位价"
    ),
)
orders = orders_full.sample(2_000, random_state=36)

marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"),
    visits=taxis["distance"],
    ad_spend=taxis["tip"],
    sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(
    min(2_000, len(marketing_full)), random_state=36
).copy()

daily = flights.assign(
    date=pd.to_datetime(
        flights["year"].astype(str) + "-" + flights["month"] + "-01"
    ),
    region="AirPassengers",
    sales=flights["passengers"],
)
print(f"样本：orders {len(orders):,} | marketing {len(marketing):,} | daily {len(daily):,} 行")


## 46.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

order = orders["category"].value_counts().index
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.countplot(data=orders, y="category", order=order, color="#1a73e8", ax=ax)
ax.set(title="各品类订单量", xlabel="订单数", ylabel="品类")
fig.tight_layout()
plt.show()


**练一练**：光看不练记不牢。下面这段代码把基础图表的柱状图方向改一下——把 `y="category"` 改成 `x="category"`，横向柱状图就会变成纵向柱状图，横轴变成品类、纵轴变成订单数。请先运行一次，观察坐标轴的变化，再在代码里自己加一行 `ax.set()` 把标题和轴标签补上。完成后再运行自检确认。


In [ ]:
# 请在下方填写代码
import matplotlib.pyplot as plt
import seaborn as sns

order = orders["category"].value_counts().index
fig, ax = plt.subplots(figsize=(8, 4.2))
# 练习：把 y="category" 改为 x="category"，让柱状图变为纵向
# TODO 补全 ax.set(title=..., xlabel=..., ylabel=...) 让读者能脱离代码读懂图
sns.countplot(
    data=orders, y="category", order=order, color="#1a73e8", ax=ax
)  # 请修改这一处
fig.tight_layout()
plt.show()


In [ ]:
# 完整参考答案
import matplotlib.pyplot as plt
import seaborn as sns

order = orders["category"].value_counts().index
fig, ax = plt.subplots(figsize=(8, 4.2))
# 把横向柱状图改为纵向：y="category" -> x="category"，并补全标题与轴标签
sns.countplot(data=orders, x="category", order=order, color="#1a73e8", ax=ax)
ax.set(title="各品类订单量", xlabel="品类", ylabel="订单数")
fig.tight_layout()
plt.show()


## 46.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.2))
sns.countplot(
    data=orders,
    x="category",
    hue="satisfied",
    stat="percent",
    palette=["#188038", "#f9ab00"],
    ax=ax,
)
ax.set(title="品类评价构成", xlabel="品类", ylabel="占全部订单比例（%）")
ax.legend(title="评价", frameon=False)
fig.tight_layout()
plt.show()


## 46.8 参数说明

- order：类别顺序
- hue：组内分类
- stat：计数或比例
- palette：颜色


## 46.9 结果解读

柱高是行数或比例，不代表金额、均值或总和。


## 46.10 本章实训：分组比较与不确定性

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="region", y="sales", ci=None, ax=ax, color="#0F766E"
)
ax.set_title("地区销售额比较")
ax.set_ylabel("销售额")
plt.show()


### 46.10.1 第一个结果怎么读

Seaborn 负责把 DataFrame 的字段映射为图形编码；先明确横轴、纵轴和每行数据的粒度，再选择图表。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
report = report.sort_values("sales", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="sales", y="region", ci=None, ax=ax, color="#F59E0B"
)
ax.set_title("按销售额排序的地区比较")
ax.set_xlabel("销售额")
ax.set_ylabel("地区")
plt.show()


### 46.10.2 第二个结果怎么读

第二个实验只改变排序和坐标方向，让读者更容易找到最大值。图表调整必须服务于阅读任务。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 46.11 错误恢复：分组字段缺失怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
required = {"region", "sales"}
missing = required - set(report.columns)
if missing:
    print("缺少字段：", sorted(missing))
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax)
    ax.set_title("地区销售额")
    plt.show()


### 46.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

绘图前先检查字段是否存在。把字段检查放在画图之前，错误会更接近真正原因，也更容易恢复。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 46.12 易错点提醒

- 把countplot误认为数值聚合
- 类别顺序随数据变化
- 类别太多导致标签拥挤


## 46.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 46.14 独立迁移练习

修改一个分组、排序或统计设置，并比较修改前后的结论。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：用 hue 按渠道分色，观察频数的构成
# 【目标】把柱子按渠道拆开，看频数和顺序之外，还能看出「构成」。
import matplotlib.pyplot as plt
import seaborn as sns

# 起点示例(已可运行)：加 hue="channel"，让每根柱子按渠道分成多段颜色。
order = orders["category"].value_counts().index
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.countplot(
    data=orders, y="category", order=order, hue="channel", palette="colorblind", ax=ax
)
ax.set(title="各品类订单量（分渠道）", xlabel="订单数", ylabel="品类")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()

# ---- 反思记录：分色后，除了频数还多看到了什么 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
import matplotlib.pyplot as plt

region_order = orders["region"].value_counts().index
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.countplot(
    data=orders,
    y="region",
    hue="channel",
    order=region_order,
    palette="colorblind",
    ax=ax,
)
ax.set(title="区域渠道订单量", xlabel="订单数", ylabel="区域")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()


## 46.15 小结

用countplot统计分类变量频数，并通过顺序和hue比较构成。


### 46.15.1 你已经掌握

- 判断频数图（countplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 46.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `order` | 类别顺序 |
| `hue` | 组内分类 |
| `stat` | 计数或比例 |
| `palette` | 颜色 |


### 46.15.3 需要注意

- 把countplot误认为数值聚合
- 类别顺序随数据变化
- 类别太多导致标签拥挤


### 46.15.4 完成检查

- [ ] 能判断什么问题适合使用频数图（countplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 46.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
